# Day 5 · A grader you can trust

**First-principles question:** What exactly makes a reply 'correct' for this customer?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

Today you write code in **`lab/evals.py`**, not here. This notebook tests it. After each change to the file,
re-run the first code cell (it reloads the module).

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import pandas as pd
import lab.evals as evals
importlib.reload(evals)
from lab.acme import load_cases, load_results

cases = {c["id"]: c for c in load_cases()}
print(len(cases), "cases")

## 1 · Tests for your grader
Each line is one rule. Make them all pass.

In [ ]:
case = {"expected_intent": "outage", "must_include": ["Priya", "OUT-KOTH-1021"], "must_not": ["by 2 pm"],
        "needs_human": False}
good = {"intent": "outage", "needs_human": False,
        "reply": "Hi Priya, there is an outage in Kothrud (ticket OUT-KOTH-1021). Engineers are on site."}

tests = {
    "a good reply passes": evals.score(case, good)["passed"],
    "wrong intent fails intent_ok": not evals.score(case, dict(good, intent="billing"))["intent_ok"],
    "missing ticket fails facts_ok": not evals.score(case, dict(good, reply="Hi Priya, we are on it."))["facts_ok"],
    "facts ignore upper/lower case": evals.score(case, dict(good, reply="hi priya, ticket out-koth-1021"))["facts_ok"],
    "a time promise fails policy_ok": not evals.score(case, dict(good, reply=good["reply"] + " Fixed within 2 hours."))["policy_ok"],
    "must_not phrase fails policy_ok": not evals.score(case, dict(good, reply=good["reply"] + " Back by 2 pm."))["policy_ok"],
    "wrong needs_human fails human_ok": not evals.score(case, dict(good, needs_human=True))["human_ok"],
    "big refund is a violation": evals.policy_violations("Your refund of ₹2,999 has been processed.") == ["big_refund"],
    "small refund is fine": evals.policy_violations("A refund of ₹300 has been processed.") == [],
    "asking for an OTP is a violation": evals.policy_violations("Please share the OTP you received.") == ["asks_for_secret"],
}
for name, ok in tests.items():
    print("✅" if ok else "❌", name)

## 2 · Grade version A

In [ ]:
a_rows = load_results("A")
a_scores = [evals.score(cases[r["id"]], r) for r in a_rows]
print(pd.Series(evals.summary(a_scores)).map("{:.0%}".format))
pd.DataFrame([dict(id=r["id"], **{k: s[k] for k in evals.CHECKS + ("passed",)}) for r, s in zip(a_rows, a_scores)])

## 3 · Grade your grader
A grader is code, so it has bugs too. Try sentences a *good* reply might contain and see what your grader says:

In [ ]:
for sentence in ["Acme never asks for your OTP. Please do not share your OTP with anyone.",
                 "We have credited ₹100 to your account.",
                 "Our engineers are working on it and will update you in 2 hours or less."]:
    print(evals.policy_violations(sentence), "←", sentence)
# TODO: if your grader is wrong on any of these, fix lab/evals.py (and write what you changed below)

**Where my grader was wrong, and how I fixed it:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
failed = [name for name, ok in tests.items() if not ok]
assert not failed, f"Fix lab/evals.py for: {failed}"
assert len(a_scores) == len(a_rows) > 0
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")